# 🫀 PulseProof — Patient-Level HPO V7, Calibrated Ensembling, Abstention & Transportability
### Byte2Beat final research notebook

This notebook is designed as a **leakage-resistant clinical-ML research study**, not merely a prediction demo.

Its central protocol is:

1. verify the patient identifier and one-to-one merge,
2. lock a **70% development / 10% calibration / 20% test** patient split,
3. compare broad model families using patient-grouped cross-validation,
4. optimize hyperparameters **only inside development patients**,
5. generate five-fold patient-grouped out-of-fold predictions for tuned models,
6. freeze the ensemble, calibration method, operating threshold, and abstention rule,
7. open the locked internal test once,
8. test the frozen source-only portable model on the second provided dataset,
9. quantify population shift rather than hiding failed transportability.

> **Research and education only. This notebook is not a medical device and must not be used for diagnosis, treatment, prognosis, screening, or emergency triage.**

## 0. Reproducibility and run budget

`competition` mode performs a broad model zoo, model-specific randomized hyperparameter search, and five-fold tuned OOF evaluation. `fast` mode is available only for debugging.

Hyperparameter search uses fixed patient-grouped folds. The locked test labels are never used for model family selection, parameter selection, blend weighting, calibration selection, threshold selection, or abstention selection.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple
import os, json, math, time, random, warnings, zipfile, hashlib, tempfile, csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from scipy.optimize import minimize
from scipy.special import logit, expit

from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import StratifiedGroupKFold, train_test_split, ParameterSampler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier

warnings.filterwarnings('ignore')

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

# Change to 'fast' only for debugging. Keep 'competition' for the final Kaggle version.
RUN_MODE = 'competition'
ENABLE_SHAP = False  # Optional; permutation importance is always produced.
TARGET_SELECTIVE_COVERAGE = 0.80

if RUN_MODE == 'competition':
    FINAL_OOF_SPLITS = 5
    HPO_SPLITS = 3
    HPO_MAX_PATIENTS = 45_000
    BOOTSTRAPS = 600
    HPO_BUDGETS = {
        'Logistic regression': 12,
        'Gaussian NB': 8,
        'Shrinkage LDA': 8,
        'Random Forest': 10,
        'Extra Trees': 12,
        'HistGradientBoosting': 18,
        'MLP': 8,
        'XGBoost': 18,
        'LightGBM': 18,
        'CatBoost': 14,
    }
else:
    FINAL_OOF_SPLITS = 3
    HPO_SPLITS = 2
    HPO_MAX_PATIENTS = 8_000
    BOOTSTRAPS = 120
    HPO_BUDGETS = {
        'Logistic regression': 2,
        'Gaussian NB': 2,
        'Shrinkage LDA': 2,
        'Random Forest': 2,
        'Extra Trees': 2,
        'HistGradientBoosting': 3,
        'MLP': 2,
        'XGBoost': 3,
        'LightGBM': 3,
        'CatBoost': 3,
    }

INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data/byte2beat_smoke_input')
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/pulseproof_v4_smoke_work')
OUTPUT_ROOT = WORK_ROOT / 'pulseproof_hpo_v7_outputs'
EXTRACT_ROOT = WORK_ROOT / 'pulseproof_hpo_v7_extracted'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print('RUN_MODE   :', RUN_MODE)
print('INPUT_ROOT :', INPUT_ROOT)
print('WORK_ROOT  :', WORK_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('HPO budgets:', HPO_BUDGETS)

## 1. Discover the Byte2Beat files

The loader searches all Kaggle input mounts and safely extracts nested ZIP archives. Precomputed PulseProof result tables are excluded so they cannot be mistaken for source data.

In [ ]:
def safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / '.complete'
    if marker.exists():
        return destination
    with zipfile.ZipFile(zip_path) as zf:
        root = destination.resolve()
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'Unsafe archive path: {member.filename}')
        zf.extractall(destination)
    marker.write_text('ok', encoding='utf-8')
    return destination

archives = sorted(INPUT_ROOT.rglob('*.zip')) if INPUT_ROOT.exists() else []
for archive in archives:
    digest = hashlib.md5(str(archive).encode()).hexdigest()[:10]
    safe_extract_zip(archive, EXTRACT_ROOT / f'{archive.stem}_{digest}')

all_roots = [INPUT_ROOT, EXTRACT_ROOT]
all_csv = []
for root in all_roots:
    if root.exists():
        all_csv.extend(root.rglob('*.csv'))

# Remove duplicate paths and any precomputed bundle result tables.
all_csv = sorted(set(p.resolve() for p in all_csv))
source_csv = [
    p for p in all_csv
    if '/assets/' not in str(p).replace('\\','/').lower()
    and 'pulseproof_outputs' not in str(p).lower()
    and 'pulseproof_byte2beat' not in str(p).lower()
]

print(f'Archives found: {len(archives)}')
print(f'Source CSV candidates: {len(source_csv)}')
for p in source_csv:
    print(' -', p)

In [ ]:
# Exact filenames are the source of truth for the official Byte2Beat folder.
# Every CSV is parsed through one delimiter/encoding-aware loader. This avoids
# a common failure mode where a semicolon/tab-delimited file is seen as one column.

CSV_SPEC_CACHE: Dict[Path, Dict] = {}
HEADER_CACHE: Dict[Path, List[str]] = {}


def normalize_column_name(value) -> str:
    # Clean transport artefacts only. Dataset-specific capitalization is restored
    # later from each role's required schema.
    return str(value).replace('\ufeff', '').strip().strip('\"').strip("'")

def _decode_sample(raw: bytes) -> Tuple[str, str]:
    # Turkish Windows exports are included as fallbacks, while utf-8-sig removes BOM.
    for encoding in ('utf-8-sig', 'utf-8', 'cp1254', 'cp1252', 'latin1'):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    return raw.decode('latin1', errors='replace'), 'latin1'


def _parse_line(line: str, sep: str) -> List[str]:
    try:
        return next(csv.reader([line], delimiter=sep))
    except Exception:
        return line.split(sep)


def detect_csv_spec(path: Path, required: set = frozenset()) -> Dict:
    path = Path(path).resolve()
    # A previously detected spec is valid for all later reads of the same file.
    if path in CSV_SPEC_CACHE:
        return CSV_SPEC_CACHE[path]

    with path.open('rb') as handle:
        raw = handle.read(2_000_000)
    text, encoding = _decode_sample(raw)
    raw_lines = text.splitlines()
    nonempty = [(i, line) for i, line in enumerate(raw_lines[:15]) if line.strip()]
    if not nonempty:
        raise ValueError(f'{path.name} appears empty.')

    required_folded = {str(c).strip().casefold() for c in required}
    candidates = []
    for sep in (',', ';', '\t', '|'):
        for original_idx, line in nonempty[:8]:
            # Some spreadsheet exports put "sep=;" on the first line.
            if line.strip().lower().startswith('sep='):
                continue
            fields = [normalize_column_name(x) for x in _parse_line(line, sep)]
            folded = {x.casefold() for x in fields}
            matched = len(required_folded & folded)
            missing = len(required_folded - folded)

            # Check whether the next available records have a compatible field count.
            following_counts = []
            for _, next_line in nonempty:
                if next_line == line or next_line.strip().lower().startswith('sep='):
                    continue
                if len(following_counts) >= 3:
                    break
                following_counts.append(len(_parse_line(next_line, sep)))
            consistency = sum(c == len(fields) for c in following_counts)

            # Required-column matches dominate; width/consistency break ties.
            score = matched * 100_000 - missing * 20_000
            score += min(len(fields), 2_000) * 5 + consistency * 200
            if len(fields) <= 1:
                score -= 50_000
            candidates.append({
                'sep': sep,
                'encoding': encoding,
                'skiprows': original_idx,
                'columns': fields,
                'matched_required': matched,
                'missing_required': missing,
                'score': score,
            })

    candidates.sort(key=lambda d: d['score'], reverse=True)
    spec = candidates[0]
    if required and spec['missing_required']:
        preview = text[:500].replace('\n', '\\n').replace('\r', '\\r')
        top = [
            {
                'sep': repr(c['sep']),
                'skiprows': c['skiprows'],
                'matched': c['matched_required'],
                'width': len(c['columns']),
                'columns': c['columns'][:12],
            }
            for c in candidates[:4]
        ]
        raise ValueError(
            f'Could not parse the expected schema from {path}. '
            f'Required={sorted(required)}; best candidates={top}; file preview={preview!r}'
        )

    CSV_SPEC_CACHE[path] = spec
    return spec


def smart_read_csv(path: Path, required: set = frozenset(), **kwargs) -> pd.DataFrame:
    path = Path(path).resolve()
    spec = detect_csv_spec(path, required)
    options = {
        'sep': spec['sep'],
        'encoding': spec['encoding'],
        'skiprows': spec['skiprows'],
        'low_memory': False,
    }
    options.update(kwargs)
    frame = pd.read_csv(path, **options)
    cleaned = [normalize_column_name(c) for c in frame.columns]
    role_canonical = {str(c).casefold(): c for c in required}
    frame.columns = [role_canonical.get(c.casefold(), c) for c in cleaned]
    if required:
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(
                f'{path.name} parsed with sep={spec["sep"]!r}, encoding={spec["encoding"]}, '
                f'but is missing columns {sorted(missing)}. Parsed columns={frame.columns.tolist()[:30]}'
            )
    return frame


def read_header(path: Path, required: set = frozenset()) -> List[str]:
    path = Path(path).resolve()
    if path not in HEADER_CACHE:
        HEADER_CACHE[path] = smart_read_csv(path, required=required, nrows=0).columns.tolist()
    return HEADER_CACHE[path]


def exact_candidates(filename: str) -> List[Path]:
    target = filename.casefold()
    matches = [p for p in source_csv if p.name.casefold() == target]
    return sorted(
        matches,
        key=lambda p: (
            0 if '/datasets/' in str(p).replace('\\','/').casefold() else 1,
            len(str(p)),
            str(p),
        ),
    )


def validate_columns(path: Path, required: set, forbidden: set = frozenset()) -> None:
    cols = set(read_header(path, required=required))
    missing = required - cols
    present_forbidden = forbidden & cols
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {sorted(missing)}')
    if present_forbidden:
        raise ValueError(f'{path.name} unexpectedly contains forbidden columns: {sorted(present_forbidden)}')


def schema_score(path: Path, required: set, preferred_tokens: Tuple[str,...]=(), forbidden: set=frozenset()) -> int:
    text = str(path).replace('\\','/').casefold()
    if 'ecg' in text or 'timeseries' in text:
        return -100_000
    try:
        cols = set(read_header(path, required=required))
    except Exception:
        return -100_000
    score = 40 * len(required & cols) - 100 * len(required - cols)
    score -= 80 * len(forbidden & cols)
    score += 12 * sum(tok.casefold() in text for tok in preferred_tokens)
    return score


def choose_exact_then_schema(
    filename: str,
    required: set,
    tokens: Tuple[str,...],
    forbidden: set=frozenset(),
) -> Tuple[Path, str, int]:
    exact = exact_candidates(filename)
    if exact:
        path = exact[0]
        validate_columns(path, required, forbidden)
        return path, 'exact filename + verified schema', 10_000

    scored = [(p, schema_score(p, required, tokens, forbidden)) for p in source_csv]
    scored.sort(key=lambda x: x[1], reverse=True)
    if not scored or scored[0][1] < 0:
        raise FileNotFoundError(
            f'Could not locate {filename}; no CSV matched required columns {sorted(required)}'
        )
    path, score = scored[0]
    validate_columns(path, required, forbidden)
    return path, 'schema fallback', score


# The official cardio_base.csv may legitimately include the `cardio` target.
# Therefore target presence is not used as a forbidden-schema signal; the base
# role is identified by its anthropometric and blood-pressure columns.
BASE_REQUIRED = {'id','age','gender','height','weight','ap_hi','ap_lo','cholesterol','smoke'}
PROCESSED_REQUIRED = {'id','gluc','alco','active','cardio'}
HEART_REQUIRED = {'Age','RestingBP','Cholesterol','HeartDisease'}

base_path, base_method, base_score = choose_exact_then_schema(
    'cardio_base.csv', BASE_REQUIRED,
    ('cardiac failure','cardio_base'),
)
processed_path, processed_method, processed_score = choose_exact_then_schema(
    'cardiac_failure_processed.csv', PROCESSED_REQUIRED,
    ('cardiac failure','processed'),
)
heart_path, heart_method, heart_score = choose_exact_then_schema(
    'heart_processed.csv', HEART_REQUIRED,
    ('heart attack','heart_processed'),
)

ecg_exact = exact_candidates('ecg_timeseries.csv')
if ecg_exact:
    ecg_path = ecg_exact[0]
    ecg_method = 'exact filename'
else:
    hinted = [p for p in source_csv if 'ecg' in str(p).casefold() or 'timeseries' in str(p).casefold()]
    ecg_path = sorted(hinted, key=lambda p: (len(str(p)), str(p)))[0] if hinted else None
    ecg_method = 'path hint' if ecg_path else 'not found'

assert base_path.resolve() != processed_path.resolve(), (
    'Data discovery selected the same CSV for base and processed tables.'
)

selection_rows = []
for role, path, method, score, required in [
    ('Cardiovascular base', base_path, base_method, base_score, BASE_REQUIRED),
    ('Labels/lifestyle', processed_path, processed_method, processed_score, PROCESSED_REQUIRED),
    ('External heart data', heart_path, heart_method, heart_score, HEART_REQUIRED),
]:
    spec = detect_csv_spec(path, required)
    selection_rows.append({
        'Role': role, 'Path': str(path), 'Selection': method, 'Score': score,
        'Separator': repr(spec['sep']), 'Encoding': spec['encoding'],
        'Header row': spec['skiprows'], 'Parsed columns': len(spec['columns']),
    })
selection_rows.append({
    'Role':'ECG timeseries','Path':str(ecg_path) if ecg_path else None,
    'Selection':ecg_method,'Score':None,'Separator':None,'Encoding':None,
    'Header row':None,'Parsed columns':None,
})
selection_manifest = pd.DataFrame(selection_rows)
display(selection_manifest)
selection_manifest.to_csv(OUTPUT_ROOT/'data_source_selection.csv', index=False)

print(f'Cardiovascular base: {base_path} ({base_method})')
print(f'Labels/lifestyle    : {processed_path} ({processed_method})')
print(f'External heart data : {heart_path} ({heart_method})')
print(f'ECG timeseries      : {ecg_path} ({ecg_method})')
for label, path, required in [
    ('base', base_path, BASE_REQUIRED),
    ('processed', processed_path, PROCESSED_REQUIRED),
    ('heart', heart_path, HEART_REQUIRED),
]:
    spec = detect_csv_spec(path, required)
    print(f'{label:9s} format: separator={spec["sep"]!r}, encoding={spec["encoding"]}, header_row={spec["skiprows"]}')


### Source-schema note

The official `cardio_base.csv` may already contain the `cardio` outcome. V7 accepts this valid schema, verifies any duplicate target column against `cardiac_failure_processed.csv`, and stops only if patient-level target values disagree.


## 2. Merge by patient and prove leakage protection

The two primary tables are merged with `validate="one_to_one"`. The notebook verifies the number of unique patient IDs and reports duplicate IDs before any model is trained.

The current dataset contains one row per patient. Therefore an ordinary row split would happen to produce the same membership behavior here, but the analysis still uses explicit patient-grouped partitions so the protocol remains correct if repeated measurements are introduced later.

In [ ]:
base = smart_read_csv(base_path, required=BASE_REQUIRED)
processed = smart_read_csv(processed_path, required=PROCESSED_REQUIRED)
heart = smart_read_csv(heart_path, required=HEART_REQUIRED)

print('base     :', base.shape)
print('processed:', processed.shape)
print('heart    :', heart.shape)

for label, frame, required in [
    ('cardio_base', base, BASE_REQUIRED),
    ('cardiac_failure_processed', processed, PROCESSED_REQUIRED),
    ('heart_processed', heart, HEART_REQUIRED),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{label} missing required columns: {sorted(missing)}')

assert base_path.resolve() != processed_path.resolve(), 'Base and processed paths must be different.'

id_audit = pd.DataFrame([
    {'Table':'cardio_base', 'Rows':len(base), 'Unique patients':base['id'].nunique(), 'Duplicate rows by id':base['id'].duplicated().sum()},
    {'Table':'processed', 'Rows':len(processed), 'Unique patients':processed['id'].nunique(), 'Duplicate rows by id':processed['id'].duplicated().sum()},
])
display(id_audit)

# Audit overlapping clinical fields before merging. The official source files may
# both contain `cardio`; that is valid and should not be treated as a file-selection error.
shared_audit_cols = [c for c in ['cardio','gluc','alco','active'] if c in base.columns and c in processed.columns]
consistency_rows = []
for col in shared_audit_cols:
    paired = base[['id', col]].merge(
        processed[['id', col]], on='id', how='inner', validate='one_to_one',
        suffixes=('_base', '_processed')
    )
    left = paired[f'{col}_base']
    right = paired[f'{col}_processed']
    equal = left.eq(right) | (left.isna() & right.isna())
    consistency_rows.append({
        'Column': col,
        'Compared patients': len(paired),
        'Mismatches': int((~equal).sum()),
        'Agreement': float(equal.mean()),
    })

consistency_audit = pd.DataFrame(consistency_rows)
if not consistency_audit.empty:
    display(consistency_audit)
    consistency_audit.to_csv(OUTPUT_ROOT/'source_column_consistency.csv', index=False)
    cardio_row = consistency_audit[consistency_audit['Column'] == 'cardio']
    if not cardio_row.empty and int(cardio_row['Mismatches'].iloc[0]) > 0:
        raise ValueError(
            'The cardio target differs between cardio_base.csv and '
            'cardiac_failure_processed.csv. Resolve the source-data inconsistency '
            'before modeling.'
        )

# Keep the base value for overlapping fields and add only fields absent from base.
extra_cols = ['id'] + [c for c in ['gluc','alco','active','cardio'] if c not in base.columns]
if len(extra_cols) > 1:
    data = base.merge(processed[extra_cols], on='id', how='inner', validate='one_to_one')
else:
    common_ids = set(base['id']) & set(processed['id'])
    data = base[base['id'].isin(common_ids)].copy()

print('Merged shape:', data.shape)
print('Target source:', 'cardio_base.csv' if 'cardio' in base.columns else 'cardiac_failure_processed.csv')

assert data['id'].nunique() == len(data), 'Expected one record per patient after one-to-one merge.'
assert 'cardio' in data.columns, 'No cardio target was found after source reconciliation.'
assert data['cardio'].isin([0,1]).all(), 'Target must be binary.'


## 3. Deterministic feature engineering and physiological quality gate

No target-driven feature selection occurs before the locked split. Derived variables are interpretable transformations of age, anthropometry, blood pressure, metabolic categories, and lifestyle indicators.

In [ ]:
data['age_years'] = data['age'] / 365.25
with np.errstate(divide='ignore', invalid='ignore'):
    data['bmi'] = data['weight'] / (data['height'] / 100.0) ** 2
    data['pulse_pressure'] = data['ap_hi'] - data['ap_lo']
    data['map'] = (data['ap_hi'] + 2 * data['ap_lo']) / 3
    data['bp_ratio'] = data['ap_hi'] / data['ap_lo'].replace(0, np.nan)

data['chol_high'] = (data['cholesterol'] > 1).astype(int)
data['gluc_high'] = (data['gluc'] > 1).astype(int)
data['hypertension_flag'] = ((data['ap_hi'] >= 140) | (data['ap_lo'] >= 90)).astype(int)
data['age_sq'] = (data['age_years'] / 10.0) ** 2
data['bmi_sq'] = (data['bmi'] / 10.0) ** 2
data['age_sbp_interaction'] = data['age_years'] * data['ap_hi'] / 1000.0
data['age_bmi_interaction'] = data['age_years'] * data['bmi'] / 1000.0
data['metabolic_interaction'] = data['cholesterol'] * data['gluc']
data['lifestyle_risk_count'] = data['smoke'] + data['alco'] + (1 - data['active'])
data['combined_risk_count'] = (
    data['chol_high'] + data['gluc_high'] + data['hypertension_flag'] +
    data['smoke'] + (1 - data['active'])
)

quality = pd.DataFrame(index=data.index)
quality['age_invalid'] = ~data['age_years'].between(18, 100)
quality['height_invalid'] = ~data['height'].between(130, 210)
quality['weight_invalid'] = ~data['weight'].between(35, 200)
quality['sbp_invalid'] = ~data['ap_hi'].between(70, 250)
quality['dbp_invalid'] = ~data['ap_lo'].between(40, 150)
quality['bp_order_invalid'] = data['ap_hi'] <= data['ap_lo']
quality['bmi_invalid'] = ~data['bmi'].between(12, 70)

data['quality_issue_count'] = quality.sum(axis=1)
clean = data.loc[data['quality_issue_count'] == 0].copy()

quality_summary = pd.DataFrame({
    'Check': quality.columns,
    'Violations': quality.sum().values,
    'Rate': quality.mean().values,
}).sort_values('Rate', ascending=False)

display(quality_summary)
print(f'Rows before checks : {len(data):,}')
print(f'Rows after checks  : {len(clean):,}')
print(f'Rejected from train: {1-len(clean)/len(data):.2%}')

plt.figure(figsize=(8,4.5))
qs = quality_summary.sort_values('Rate')
plt.barh(qs['Check'], qs['Rate']*100)
plt.xlabel('Records failing check (%)')
plt.title('Physiological data-quality violations')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'01_quality_gate.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Locked patient-level 70/10/20 partition

- **70% development patients:** model screening, hyperparameter search, tuned OOF estimation, and blend weighting
- **10% calibration patients:** calibration method, operating threshold, and abstention rule
- **20% locked test patients:** opened once after all choices are frozen

Every split is stratified by the patient-level target, and patient overlap must equal zero.

In [ ]:
FEATURES = [
    'age_years','gender','height','weight','ap_hi','ap_lo',
    'cholesterol','gluc','smoke','alco','active','bmi',
    'pulse_pressure','map','bp_ratio','chol_high','gluc_high','hypertension_flag',
    'age_sq','bmi_sq','age_sbp_interaction','age_bmi_interaction',
    'metabolic_interaction','lifestyle_risk_count','combined_risk_count'
]
TARGET = 'cardio'
GROUP = 'id'

patient_table = clean.groupby(GROUP, as_index=False).agg(target=(TARGET,'max'), rows=(TARGET,'size'))

traincal_ids, test_ids = train_test_split(
    patient_table[GROUP], test_size=0.20,
    stratify=patient_table['target'], random_state=SEED
)
traincal_table = patient_table[patient_table[GROUP].isin(traincal_ids)]
train_ids, cal_ids = train_test_split(
    traincal_table[GROUP], test_size=0.125,
    stratify=traincal_table['target'], random_state=SEED+1
)

train_df = clean[clean[GROUP].isin(set(train_ids))].copy()
cal_df = clean[clean[GROUP].isin(set(cal_ids))].copy()
test_df = clean[clean[GROUP].isin(set(test_ids))].copy()

split_sets = {
    'Development': set(train_df[GROUP]),
    'Calibration': set(cal_df[GROUP]),
    'Locked test': set(test_df[GROUP]),
}

overlap = pd.DataFrame([
    {'Pair':'Development ∩ Calibration','Patient overlap':len(split_sets['Development'] & split_sets['Calibration'])},
    {'Pair':'Development ∩ Test','Patient overlap':len(split_sets['Development'] & split_sets['Locked test'])},
    {'Pair':'Calibration ∩ Test','Patient overlap':len(split_sets['Calibration'] & split_sets['Locked test'])},
])

split_summary = pd.DataFrame([
    {'Split':name, 'Rows':len(df), 'Patients':df[GROUP].nunique(), 'Prevalence':df[TARGET].mean()}
    for name,df in [('Development',train_df),('Calibration',cal_df),('Locked test',test_df)]
])

display(split_summary)
display(overlap)
assert overlap['Patient overlap'].sum() == 0

plt.figure(figsize=(7.5,4.5))
plt.bar(split_summary['Split'], split_summary['Patients'])
for i,v in enumerate(split_summary['Patients']):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.ylabel('Unique patients')
plt.title('Locked patient-level evaluation partitions')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'02_patient_level_split.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Broad model zoo and model-specific search spaces

The notebook covers statistical, bagged-tree, neural-network, and boosted-tree families. Optional Kaggle libraries are detected automatically.

Hyperparameter search is intentionally model-specific. XGBoost, LightGBM, and CatBoost expose different controls, so a single generic grid would be scientifically weak and computationally wasteful.

In [ ]:
OPTIONAL = {}
try:
    from xgboost import XGBClassifier
    OPTIONAL['XGBoost'] = True
except Exception as e:
    OPTIONAL['XGBoost'] = False
    print('XGBoost unavailable:', e)
try:
    from lightgbm import LGBMClassifier
    OPTIONAL['LightGBM'] = True
except Exception as e:
    OPTIONAL['LightGBM'] = False
    print('LightGBM unavailable:', e)
try:
    from catboost import CatBoostClassifier
    OPTIONAL['CatBoost'] = True
except Exception as e:
    OPTIONAL['CatBoost'] = False
    print('CatBoost unavailable:', e)

class CatBoostCompat(BaseEstimator, ClassifierMixin):
    """Cloneable CatBoost adapter with fold-local median imputation."""
    def __init__(self, iterations=550, depth=6, learning_rate=0.035,
                 l2_leaf_reg=5.0, random_strength=1.0,
                 bagging_temperature=1.0, border_count=128,
                 random_seed=SEED):
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.random_strength = random_strength
        self.bagging_temperature = bagging_temperature
        self.border_count = border_count
        self.random_seed = random_seed

    def fit(self, X, y):
        self.imputer_ = SimpleImputer(strategy='median')
        Xt = self.imputer_.fit_transform(X)
        self.model_ = CatBoostClassifier(
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            l2_leaf_reg=self.l2_leaf_reg,
            random_strength=self.random_strength,
            bagging_temperature=self.bagging_temperature,
            border_count=self.border_count,
            loss_function='Logloss',
            eval_metric='AUC',
            random_seed=self.random_seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )
        self.model_.fit(Xt, y)
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self.imputer_.transform(X))


def scaled(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', estimator),
    ])


def unscaled(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', estimator),
    ])

BASE_MODELS: Dict[str, object] = {
    'Dummy prevalence': unscaled(DummyClassifier(strategy='prior')),
    'Logistic regression': scaled(LogisticRegression(max_iter=2500, C=0.5, random_state=SEED)),
    'Gaussian NB': scaled(GaussianNB(var_smoothing=1e-8)),
    'Shrinkage LDA': scaled(LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
    'Random Forest': unscaled(RandomForestClassifier(
        n_estimators=400, min_samples_leaf=10, max_features='sqrt',
        class_weight=None, n_jobs=-1, random_state=SEED
    )),
    'Extra Trees': unscaled(ExtraTreesClassifier(
        n_estimators=500, min_samples_leaf=8, max_features=0.8,
        class_weight=None, n_jobs=-1, random_state=SEED
    )),
    'HistGradientBoosting': unscaled(HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=350, max_leaf_nodes=24,
        min_samples_leaf=30, l2_regularization=1.5, random_state=SEED
    )),
    'MLP': scaled(MLPClassifier(
        hidden_layer_sizes=(64, 32), alpha=0.002, learning_rate_init=0.001,
        early_stopping=True, validation_fraction=0.12,
        max_iter=220, random_state=SEED
    )),
}

if OPTIONAL.get('XGBoost'):
    BASE_MODELS['XGBoost'] = unscaled(XGBClassifier(
        n_estimators=550, max_depth=4, learning_rate=0.035,
        min_child_weight=8, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.10, reg_lambda=2.0, gamma=0.0,
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', n_jobs=-1, random_state=SEED
    ))
if OPTIONAL.get('LightGBM'):
    BASE_MODELS['LightGBM'] = unscaled(LGBMClassifier(
        n_estimators=550, learning_rate=0.035, num_leaves=24,
        max_depth=-1, min_child_samples=35, subsample=0.85,
        colsample_bytree=0.85, reg_alpha=0.10, reg_lambda=2.0,
        n_jobs=-1, random_state=SEED, verbosity=-1
    ))
if OPTIONAL.get('CatBoost'):
    BASE_MODELS['CatBoost'] = CatBoostCompat(
        iterations=550, depth=6, learning_rate=0.035,
        l2_leaf_reg=5.0, random_seed=SEED
    )

PARAM_SPACES = {
    'Logistic regression': {
        'model__C': np.logspace(-3, 2, 28).tolist(),
        'model__class_weight': [None, 'balanced'],
    },
    'Gaussian NB': {
        'model__var_smoothing': np.logspace(-12, -6, 36).tolist(),
    },
    'Shrinkage LDA': {
        'model__shrinkage': ['auto'] + np.linspace(0.0, 1.0, 21).tolist(),
    },
    'Random Forest': {
        'model__n_estimators': [300, 500, 700],
        'model__max_depth': [None, 6, 10, 16, 24],
        'model__min_samples_leaf': [2, 5, 10, 20, 40],
        'model__max_features': ['sqrt', 0.5, 0.8, 1.0],
        'model__max_samples': [None, 0.70, 0.90],
        'model__class_weight': [None, 'balanced', 'balanced_subsample'],
    },
    'Extra Trees': {
        'model__n_estimators': [350, 550, 750],
        'model__max_depth': [None, 8, 12, 18, 26],
        'model__min_samples_leaf': [2, 5, 8, 15, 30],
        'model__max_features': ['sqrt', 0.5, 0.8, 1.0],
        'model__class_weight': [None, 'balanced'],
    },
    'HistGradientBoosting': {
        'model__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.12],
        'model__max_iter': [220, 350, 500],
        'model__max_leaf_nodes': [7, 15, 24, 31, 63],
        'model__max_depth': [None, 3, 5, 7],
        'model__min_samples_leaf': [10, 20, 30, 50, 80],
        'model__l2_regularization': [0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
        'model__max_bins': [64, 128, 255],
    },
    'MLP': {
        'model__hidden_layer_sizes': [(32,), (64,), (64,32), (128,64), (128,64,32)],
        'model__alpha': np.logspace(-5, -2, 10).tolist(),
        'model__learning_rate_init': [0.0003, 0.0007, 0.001, 0.002],
        'model__activation': ['relu', 'tanh'],
        'model__batch_size': [128, 256, 512],
    },
}

if OPTIONAL.get('XGBoost'):
    PARAM_SPACES['XGBoost'] = {
        'model__n_estimators': [300, 450, 600, 800],
        'model__max_depth': [2, 3, 4, 5, 6],
        'model__min_child_weight': [1, 3, 5, 8, 12],
        'model__learning_rate': [0.02, 0.03, 0.05, 0.08],
        'model__subsample': [0.70, 0.85, 1.0],
        'model__colsample_bytree': [0.70, 0.85, 1.0],
        'model__reg_alpha': [0.0, 0.05, 0.10, 0.50, 1.0],
        'model__reg_lambda': [1.0, 2.0, 5.0, 10.0],
        'model__gamma': [0.0, 0.05, 0.10, 0.50],
    }
if OPTIONAL.get('LightGBM'):
    PARAM_SPACES['LightGBM'] = {
        'model__n_estimators': [300, 450, 600, 800],
        'model__learning_rate': [0.02, 0.03, 0.05, 0.08],
        'model__num_leaves': [7, 15, 24, 31, 47, 63],
        'model__max_depth': [-1, 3, 5, 7, 10],
        'model__min_child_samples': [10, 20, 35, 60, 100],
        'model__subsample': [0.70, 0.85, 1.0],
        'model__colsample_bytree': [0.70, 0.85, 1.0],
        'model__reg_alpha': [0.0, 0.05, 0.10, 0.50, 1.0],
        'model__reg_lambda': [0.0, 1.0, 2.0, 5.0, 10.0],
        'model__min_split_gain': [0.0, 0.01, 0.05, 0.10],
    }
if OPTIONAL.get('CatBoost'):
    PARAM_SPACES['CatBoost'] = {
        'iterations': [350, 500, 700, 900],
        'depth': [4, 5, 6, 7, 8],
        'learning_rate': [0.02, 0.03, 0.05, 0.08],
        'l2_leaf_reg': [1.0, 3.0, 5.0, 8.0, 12.0],
        'random_strength': [0.0, 0.5, 1.0, 2.0],
        'bagging_temperature': [0.0, 0.5, 1.0, 2.0],
        'border_count': [64, 128, 254],
    }

print('Models available:', len(BASE_MODELS))
for name in BASE_MODELS:
    print(' -', name, '| HPO trials:', HPO_BUDGETS.get(name, 0))

## 6. Metrics and a stratified patient subset for HPO

A bounded HPO subset reduces computational waste while preserving patient-level class balance. The final tuned OOF benchmark is always performed on **all development patients**.

The fixed HPO objective rewards discrimination and probability quality:

- 50% ROC-AUC
- 25% PR-AUC
- 15% inverse Brier score
- 10% inverse ECE

In [ ]:
def ece_score(y_true, p, n_bins=12):
    y_true = np.asarray(y_true)
    p = np.clip(np.asarray(p), 1e-7, 1-1e-7)
    edges = np.linspace(0,1,n_bins+1)
    total = 0.0
    for i in range(n_bins):
        hi = edges[i+1] + (1e-12 if i == n_bins-1 else 0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.any():
            total += mask.mean() * abs(y_true[mask].mean() - p[mask].mean())
    return float(total)


def metric_row(y, p, threshold=0.5):
    y = np.asarray(y)
    p = np.clip(np.asarray(p), 1e-7, 1-1e-7)
    pred = (p >= threshold).astype(int)
    return {
        'ROC-AUC': roc_auc_score(y,p),
        'PR-AUC': average_precision_score(y,p),
        'Brier': brier_score_loss(y,p),
        'LogLoss': log_loss(y,p),
        'ECE': ece_score(y,p),
        'Accuracy': accuracy_score(y,pred),
        'Balanced accuracy': balanced_accuracy_score(y,pred),
        'F1': f1_score(y,pred,zero_division=0),
    }


def hpo_objective(metrics):
    return (
        0.50 * metrics['ROC-AUC'] +
        0.25 * metrics['PR-AUC'] +
        0.15 * (1.0 - metrics['Brier']) +
        0.10 * (1.0 - metrics['ECE'])
    )


def bootstrap_metric_ci(y, p, metric, n_boot=BOOTSTRAPS, seed=SEED):
    y=np.asarray(y); p=np.asarray(p); rng=np.random.default_rng(seed); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(y),len(y))
        if np.unique(y[idx]).size<2:
            continue
        vals.append(metric(y[idx],p[idx]))
    return np.quantile(vals,[0.025,0.975]) if vals else (np.nan,np.nan)

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int).to_numpy()
g_train = train_df[GROUP].to_numpy()

if len(train_df) > HPO_MAX_PATIENTS:
    hpo_idx, _ = train_test_split(
        np.arange(len(train_df)), train_size=HPO_MAX_PATIENTS,
        stratify=y_train, random_state=SEED+7
    )
else:
    hpo_idx = np.arange(len(train_df))

X_hpo = X_train.iloc[hpo_idx].reset_index(drop=True)
y_hpo = y_train[hpo_idx]
g_hpo = g_train[hpo_idx]

hpo_cv = list(StratifiedGroupKFold(
    n_splits=HPO_SPLITS, shuffle=True, random_state=SEED+8
).split(X_hpo, y_hpo, g_hpo))

hpo_overlap = sum(len(set(g_hpo[tr]) & set(g_hpo[va])) for tr,va in hpo_cv)
assert hpo_overlap == 0
print(f'HPO patients: {len(X_hpo):,}; folds: {HPO_SPLITS}; patient overlap: {hpo_overlap}')

## 7. Baseline model screen and randomized hyperparameter optimization

Every family is first evaluated with its declared baseline. Tunable families then receive a reproducible randomized search. Failed configurations are recorded rather than silently discarded.

The search history itself is part of the research output: it reveals whether gains are robust or merely the result of one lucky configuration.

In [ ]:
def grouped_oof_for_estimator(estimator, X, y, groups, splits):
    oof = np.full(len(X), np.nan)
    fold_rows = []
    for fold, (tr, va) in enumerate(splits, 1):
        assert len(set(groups[tr]) & set(groups[va])) == 0
        model = clone(estimator)
        t0 = time.time()
        model.fit(X.iloc[tr], y[tr])
        p = model.predict_proba(X.iloc[va])[:,1]
        oof[va] = p
        row = {'Fold': fold, 'Seconds': time.time()-t0}
        row.update(metric_row(y[va], p))
        fold_rows.append(row)
    if not np.isfinite(oof).all():
        raise RuntimeError('OOF predictions contain non-finite values.')
    return oof, pd.DataFrame(fold_rows)


def jsonable_params(params):
    cleaned = {}
    for k,v in params.items():
        if isinstance(v, (np.integer,)):
            v = int(v)
        elif isinstance(v, (np.floating,)):
            v = float(v)
        elif isinstance(v, tuple):
            v = list(v)
        cleaned[k] = v
    return cleaned

hpo_records = []
best_params = {}
baseline_rows = []
hpo_start = time.time()

for model_no, (name, base_estimator) in enumerate(BASE_MODELS.items(), 1):
    print(f'\n[{model_no}/{len(BASE_MODELS)}] {name}')
    candidates = [({}, 'baseline')]
    budget = HPO_BUDGETS.get(name, 0)
    if name in PARAM_SPACES and budget > 0:
        sampled = list(ParameterSampler(PARAM_SPACES[name], n_iter=budget, random_state=SEED+model_no))
        candidates.extend((p, 'randomized') for p in sampled)

    family_rows = []
    for trial_no, (params, source) in enumerate(candidates):
        try:
            estimator = clone(base_estimator).set_params(**params)
            t0 = time.time()
            oof, fold_df = grouped_oof_for_estimator(estimator, X_hpo, y_hpo, g_hpo, hpo_cv)
            metrics = metric_row(y_hpo, oof)
            objective = hpo_objective(metrics)
            record = {
                'Model': name, 'Trial': trial_no, 'Source': source,
                'Status': 'ok', 'Objective': objective,
                'Runtime seconds': time.time()-t0,
                'Parameters': json.dumps(jsonable_params(params), sort_keys=True),
                **metrics,
            }
        except Exception as e:
            record = {
                'Model': name, 'Trial': trial_no, 'Source': source,
                'Status': 'failed', 'Objective': np.nan,
                'Runtime seconds': np.nan,
                'Parameters': json.dumps(jsonable_params(params), sort_keys=True),
                'Error': f'{type(e).__name__}: {str(e)[:250]}',
            }
        hpo_records.append(record)
        family_rows.append(record)
        if record['Status'] == 'ok':
            print(f"  trial {trial_no:02d} | objective={record['Objective']:.5f} | AUC={record['ROC-AUC']:.5f}")
        else:
            print(f"  trial {trial_no:02d} | FAILED | {record.get('Error','')}")

    family_ok = pd.DataFrame(family_rows)
    family_ok = family_ok[family_ok.Status == 'ok'].sort_values(
        ['Objective','ROC-AUC','Brier'], ascending=[False,False,True]
    )
    if family_ok.empty:
        raise RuntimeError(f'All configurations failed for {name}')

    winner = family_ok.iloc[0]
    best_params[name] = json.loads(winner['Parameters'])
    baseline = family_ok[family_ok.Source == 'baseline'].iloc[0]
    baseline_rows.append({
        'Model': name,
        'Baseline objective': baseline['Objective'],
        'Baseline ROC-AUC': baseline['ROC-AUC'],
        'Baseline PR-AUC': baseline['PR-AUC'],
        'Baseline Brier': baseline['Brier'],
        'Tuned objective': winner['Objective'],
        'Tuned ROC-AUC': winner['ROC-AUC'],
        'Tuned PR-AUC': winner['PR-AUC'],
        'Tuned Brier': winner['Brier'],
        'Objective gain': winner['Objective'] - baseline['Objective'],
        'AUC gain': winner['ROC-AUC'] - baseline['ROC-AUC'],
        'Best parameters': winner['Parameters'],
    })

hpo_trials = pd.DataFrame(hpo_records)
hpo_summary = pd.DataFrame(baseline_rows).sort_values('Tuned objective', ascending=False)

print(f'\nHPO time: {(time.time()-hpo_start)/60:.1f} minutes')
display(hpo_summary)

hpo_trials.to_csv(OUTPUT_ROOT/'hpo_all_trials.csv', index=False)
hpo_summary.to_csv(OUTPUT_ROOT/'hpo_baseline_vs_tuned.csv', index=False)
with open(OUTPUT_ROOT/'hpo_best_parameters.json','w',encoding='utf-8') as f:
    json.dump(best_params, f, indent=2)

# Baseline vs tuned AUC.
plot = hpo_summary[hpo_summary.Model != 'Dummy prevalence'].sort_values('Tuned ROC-AUC')
ypos = np.arange(len(plot))
plt.figure(figsize=(10,7))
plt.hlines(ypos, plot['Baseline ROC-AUC'], plot['Tuned ROC-AUC'], linewidth=2)
plt.scatter(plot['Baseline ROC-AUC'], ypos, label='Baseline')
plt.scatter(plot['Tuned ROC-AUC'], ypos, label='Tuned')
plt.yticks(ypos, plot['Model'])
plt.xlabel('Patient-grouped HPO ROC-AUC')
plt.title('Hyperparameter optimization gain by model family')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'03_hpo_baseline_vs_tuned.png', dpi=180, bbox_inches='tight')
plt.show()

# Best-so-far convergence for the most extensively searched families.
plt.figure(figsize=(10,6))
for name in hpo_summary.Model:
    d = hpo_trials[(hpo_trials.Model==name) & (hpo_trials.Status=='ok')].sort_values('Trial')
    if len(d) < 4:
        continue
    plt.plot(d['Trial'], d['Objective'].cummax(), marker='o', markersize=3, label=name)
plt.xlabel('Trial number')
plt.ylabel('Best composite HPO objective so far')
plt.title('Hyperparameter search convergence')
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'04_hpo_convergence.png', dpi=180, bbox_inches='tight')
plt.show()

## 8. Five-fold tuned patient-grouped OOF benchmark

The best parameters from HPO are now evaluated on **all development patients** under five non-overlapping patient folds. These tuned OOF predictions are used for model ranking and ensemble weighting.

In [ ]:
TUNED_MODELS = {}
for name, base in BASE_MODELS.items():
    TUNED_MODELS[name] = clone(base).set_params(**best_params.get(name, {}))

final_cv = StratifiedGroupKFold(n_splits=FINAL_OOF_SPLITS, shuffle=True, random_state=SEED+21)
final_splits = list(final_cv.split(X_train, y_train, g_train))

OOF = {}
fold_records = []
fold_integrity = []
start = time.time()

for model_idx, (name, estimator) in enumerate(TUNED_MODELS.items(), 1):
    print(f'[{model_idx}/{len(TUNED_MODELS)}] tuned {name}')
    oof = np.full(len(train_df), np.nan)
    for fold, (tr, va) in enumerate(final_splits, 1):
        overlap_count = len(set(g_train[tr]) & set(g_train[va]))
        fold_integrity.append({'Model':name,'Fold':fold,'Patient overlap':overlap_count})
        assert overlap_count == 0
        model = clone(estimator)
        t0 = time.time()
        model.fit(X_train.iloc[tr], y_train[tr])
        p = model.predict_proba(X_train.iloc[va])[:,1]
        oof[va] = p
        row = {'Model':name,'Fold':fold,'Seconds':time.time()-t0}
        row.update(metric_row(y_train[va],p))
        fold_records.append(row)
    assert np.isfinite(oof).all()
    OOF[name] = oof

print(f'Tuned full-development OOF time: {(time.time()-start)/60:.1f} minutes')
assert pd.DataFrame(fold_integrity)['Patient overlap'].sum() == 0

fold_metrics = pd.DataFrame(fold_records)
benchmark = []
for name,p in OOF.items():
    row = {'Model':name}
    row.update(metric_row(y_train,p))
    lo,hi = bootstrap_metric_ci(y_train,p,roc_auc_score,n_boot=BOOTSTRAPS,seed=SEED+22)
    row['AUC CI low'] = lo
    row['AUC CI high'] = hi
    row['Fold AUC SD'] = fold_metrics.loc[fold_metrics.Model==name,'ROC-AUC'].std()
    row['Runtime seconds'] = fold_metrics.loc[fold_metrics.Model==name,'Seconds'].sum()
    row['Tuned parameters'] = json.dumps(best_params.get(name, {}), sort_keys=True)
    benchmark.append(row)

benchmark = pd.DataFrame(benchmark).sort_values(
    ['ROC-AUC','PR-AUC','Brier','ECE'], ascending=[False,False,True,True]
).reset_index(drop=True)
benchmark['Selection rank'] = np.arange(1,len(benchmark)+1)
display(benchmark)

benchmark.to_csv(OUTPUT_ROOT/'tuned_model_oof_benchmark.csv',index=False)
fold_metrics.to_csv(OUTPUT_ROOT/'tuned_model_fold_metrics.csv',index=False)
pd.DataFrame(fold_integrity).to_csv(OUTPUT_ROOT/'patient_fold_integrity.csv',index=False)

plot_df = benchmark[benchmark.Model!='Dummy prevalence'].sort_values('ROC-AUC')
plt.figure(figsize=(10,7))
xerr = np.vstack([
    plot_df['ROC-AUC']-plot_df['AUC CI low'],
    plot_df['AUC CI high']-plot_df['ROC-AUC']
])
plt.errorbar(plot_df['ROC-AUC'], plot_df['Model'], xerr=xerr, fmt='o', capsize=3)
plt.xlabel('Tuned patient-grouped OOF ROC-AUC (95% bootstrap CI)')
plt.title('Optimized model families under identical patient folds')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'05_tuned_model_zoo_auc_ci.png',dpi=180,bbox_inches='tight')
plt.show()

# Probability diversity matters for blending.
top_names = benchmark.loc[benchmark.Model!='Dummy prevalence','Model'].head(min(8,len(benchmark)-1)).tolist()
corr = pd.DataFrame({n:OOF[n] for n in top_names}).corr()
plt.figure(figsize=(8,7))
plt.imshow(corr.values,vmin=max(0.75,float(corr.values.min())),vmax=1,cmap='viridis')
plt.xticks(range(len(corr)),corr.columns,rotation=60,ha='right')
plt.yticks(range(len(corr)),corr.index)
plt.colorbar(label='OOF probability correlation')
plt.title('Tuned model diversity')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'06_tuned_model_correlation.png',dpi=180,bbox_inches='tight')
plt.show()

## 9. Freeze the tuned ensemble and calibration strategy

Nonnegative ensemble weights are optimized on development OOF predictions—not on the locked test. Calibration methods are compared only within calibration patients. The final binary operating threshold is also selected on calibration patients and then frozen.

In [ ]:
nontrivial = benchmark[benchmark.Model!='Dummy prevalence']
selected_names = nontrivial['Model'].head(min(6,len(nontrivial))).tolist()
print('Frozen candidate models:', selected_names)

X_cal = cal_df[FEATURES]
y_cal = cal_df[TARGET].astype(int).to_numpy()
X_test = test_df[FEATURES]
y_test = test_df[TARGET].astype(int).to_numpy()

fitted = {}
P_cal = {}
P_test = {}
for name in selected_names:
    model = clone(TUNED_MODELS[name])
    model.fit(X_train,y_train)
    fitted[name] = model
    P_cal[name] = model.predict_proba(X_cal)[:,1]
    P_test[name] = model.predict_proba(X_test)[:,1]

Pcal = pd.DataFrame(P_cal)
Ptest = pd.DataFrame(P_test)
Poof = pd.DataFrame({n: OOF[n] for n in selected_names})


def optimize_weights(P,y):
    P=np.asarray(P); y=np.asarray(y); k=P.shape[1]
    def objective(w):
        pred=np.clip(P@w,1e-7,1-1e-7)
        return 0.60*log_loss(y,pred)+0.40*brier_score_loss(y,pred)+0.002*np.sum((w-1/k)**2)
    res=minimize(
        objective,np.ones(k)/k,bounds=[(0,1)]*k,
        constraints={'type':'eq','fun':lambda w:w.sum()-1},method='SLSQP'
    )
    return res.x if res.success else np.ones(k)/k


def fit_calibration_spec(method,p,y):
    p=np.clip(np.asarray(p),1e-6,1-1e-6)
    y=np.asarray(y)
    if method=='none':
        return {'method':'none'}
    if method=='sigmoid':
        m=LogisticRegression(C=1e3,max_iter=1000)
        m.fit(logit(p).reshape(-1,1),y)
        return {
            'method':'sigmoid',
            'coef':float(m.coef_.ravel()[0]),
            'intercept':float(m.intercept_.ravel()[0]),
        }
    iso=IsotonicRegression(out_of_bounds='clip')
    iso.fit(p,y)
    return {
        'method':'isotonic',
        'x':iso.X_thresholds_.astype(float).tolist(),
        'y':iso.y_thresholds_.astype(float).tolist(),
    }


def apply_calibration(spec,p):
    p=np.clip(np.asarray(p),1e-6,1-1e-6)
    if spec['method']=='none':
        return p
    if spec['method']=='sigmoid':
        return expit(spec['coef']*logit(p)+spec['intercept'])
    return np.clip(np.interp(p,np.asarray(spec['x']),np.asarray(spec['y'])),1e-6,1-1e-6)

optimized_weights = optimize_weights(Poof.values,y_train)
blend_defs = {
    'Best tuned single': np.eye(len(selected_names))[0],
    'Equal tuned top 3': np.r_[np.ones(min(3,len(selected_names)))/min(3,len(selected_names)),np.zeros(max(0,len(selected_names)-3))],
    'Equal tuned top 6': np.ones(len(selected_names))/len(selected_names),
    'OOF-optimized nonnegative': optimized_weights,
}

meta_fit_idx,meta_val_idx = train_test_split(
    np.arange(len(cal_df)),test_size=0.40,stratify=y_cal,random_state=SEED+23
)

meta_results=[]
for blend_name,w in blend_defs.items():
    raw_fit=Pcal.iloc[meta_fit_idx].values@w
    raw_val=Pcal.iloc[meta_val_idx].values@w
    for method in ['none','sigmoid','isotonic']:
        spec=fit_calibration_spec(method,raw_fit,y_cal[meta_fit_idx])
        pv=apply_calibration(spec,raw_val)
        row={'Blend':blend_name,'Calibration':method}
        row.update(metric_row(y_cal[meta_val_idx],pv))
        meta_results.append(row)

meta_results=pd.DataFrame(meta_results)
meta_results['Joint rank']=(
    meta_results['ROC-AUC'].rank(ascending=False,method='min')+
    meta_results['PR-AUC'].rank(ascending=False,method='min')+
    meta_results['Brier'].rank(ascending=True,method='min')+
    meta_results['ECE'].rank(ascending=True,method='min')
)
meta_results=meta_results.sort_values(['Joint rank','Brier','ROC-AUC'],ascending=[True,True,False])
display(meta_results)

chosen=meta_results.iloc[0]
CHOSEN_BLEND=str(chosen['Blend'])
CHOSEN_CAL=str(chosen['Calibration'])
final_weights=blend_defs[CHOSEN_BLEND]

raw_cal=Pcal.values@final_weights
raw_test=Ptest.values@final_weights
calibration_spec=fit_calibration_spec(CHOSEN_CAL,raw_cal,y_cal)
p_cal_final=apply_calibration(calibration_spec,raw_cal)
p_test_final=apply_calibration(calibration_spec,raw_test)

# Freeze the binary operating threshold on calibration patients.
threshold_rows=[]
for threshold in np.linspace(0.10,0.90,321):
    row={'Threshold':threshold}
    row.update(metric_row(y_cal,p_cal_final,threshold=threshold))
    threshold_rows.append(row)
threshold_table=pd.DataFrame(threshold_rows)
best_bal=threshold_table['Balanced accuracy'].max()
threshold_candidates=threshold_table[np.isclose(threshold_table['Balanced accuracy'],best_bal)]
DECISION_THRESHOLD=float(threshold_candidates.iloc[(threshold_candidates['Threshold']-0.5).abs().argmin()]['Threshold'])

print('Chosen blend:',CHOSEN_BLEND)
print('Chosen calibration:',CHOSEN_CAL)
print('Frozen decision threshold:',DECISION_THRESHOLD)

weight_table=pd.DataFrame({'Model':selected_names,'Weight':final_weights}).sort_values('Weight',ascending=False)
display(weight_table)

weight_table.to_csv(OUTPUT_ROOT/'ensemble_weights.csv',index=False)
meta_results.to_csv(OUTPUT_ROOT/'ensemble_calibration_selection.csv',index=False)
threshold_table.to_csv(OUTPUT_ROOT/'calibration_threshold_search.csv',index=False)

plt.figure(figsize=(8,4.8))
wplot=weight_table.sort_values('Weight')
plt.barh(wplot['Model'],wplot['Weight'])
plt.xlabel('Frozen ensemble weight')
plt.title('OOF-derived nonnegative ensemble')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'07_ensemble_weights.png',dpi=180,bbox_inches='tight')
plt.show()

## 10. One-time locked internal test evaluation

No model family, hyperparameter, feature, blend weight, calibration method, decision threshold, or abstention threshold is selected using these labels.

In [ ]:
final_metrics=metric_row(y_test,p_test_final,threshold=DECISION_THRESHOLD)
auc_ci=bootstrap_metric_ci(y_test,p_test_final,roc_auc_score,n_boot=BOOTSTRAPS,seed=SEED+31)
pr_ci=bootstrap_metric_ci(y_test,p_test_final,average_precision_score,n_boot=BOOTSTRAPS,seed=SEED+32)

final_test_table=pd.DataFrame([{
    **final_metrics,
    'Decision threshold':DECISION_THRESHOLD,
    'ROC-AUC CI low':auc_ci[0],'ROC-AUC CI high':auc_ci[1],
    'PR-AUC CI low':pr_ci[0],'PR-AUC CI high':pr_ci[1],
    'N patients':len(y_test),
    'Blend':CHOSEN_BLEND,'Calibration':CHOSEN_CAL,
}])
display(final_test_table)
final_test_table.to_csv(OUTPUT_ROOT/'locked_test_metrics.csv',index=False)

fpr,tpr,_=roc_curve(y_test,p_test_final)
prec,rec,_=precision_recall_curve(y_test,p_test_final)
fig,ax=plt.subplots(1,2,figsize=(12,4.8))
ax[0].plot(fpr,tpr,label=f'AUC {final_metrics["ROC-AUC"]:.3f}')
ax[0].plot([0,1],[0,1],'--',linewidth=1)
ax[0].set(xlabel='False positive rate',ylabel='True positive rate',title='Locked-test ROC curve')
ax[0].legend()
ax[1].plot(rec,prec,label=f'AP {final_metrics["PR-AUC"]:.3f}')
ax[1].axhline(y_test.mean(),linestyle='--',linewidth=1,label='Prevalence')
ax[1].set(xlabel='Recall',ylabel='Precision',title='Locked-test precision–recall curve')
ax[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'08_locked_test_roc_pr.png',dpi=180,bbox_inches='tight')
plt.show()

## 11. Reliability diagram and exploratory decision-curve analysis

Decision-curve analysis is presented as an exploratory model-behavior analysis. It is not evidence of clinical utility because the dataset does not provide a validated intervention pathway or harm-benefit specification.

In [ ]:
def calibration_df(y,p,n_bins=12):
    edges=np.quantile(p,np.linspace(0,1,n_bins+1))
    edges=np.unique(edges)
    bins=pd.cut(p,bins=edges,include_lowest=True,duplicates='drop')
    return pd.DataFrame({'y':y,'p':p,'bin':bins}).groupby('bin',observed=True).agg(
        n=('y','size'), predicted=('p','mean'), observed=('y','mean')
    ).reset_index(drop=True)

cal_table=calibration_df(y_test,p_test_final)
display(cal_table)
plt.figure(figsize=(6,5.5))
plt.plot([0,1],[0,1],'--',label='Perfect calibration')
plt.plot(cal_table['predicted'],cal_table['observed'],'o-',label=f'PulseProof (ECE={final_metrics["ECE"]:.3f})')
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed frequency')
plt.title('Locked-test reliability diagram'); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'09_locked_test_calibration.png',dpi=180,bbox_inches='tight')
plt.show()

thresholds=np.linspace(0.05,0.80,76)
n=len(y_test); prevalence=y_test.mean(); rows=[]
for pt in thresholds:
    pred=p_test_final>=pt
    tp=((pred==1)&(y_test==1)).sum(); fp=((pred==1)&(y_test==0)).sum()
    nb=tp/n - fp/n*(pt/(1-pt))
    all_nb=prevalence-(1-prevalence)*(pt/(1-pt))
    rows.append({'Threshold':pt,'PulseProof':nb,'Treat all':all_nb,'Treat none':0.0})
dca=pd.DataFrame(rows)
plt.figure(figsize=(8,5))
plt.plot(dca['Threshold'],dca['PulseProof'],label='PulseProof')
plt.plot(dca['Threshold'],dca['Treat all'],label='Treat all',linestyle='--')
plt.plot(dca['Threshold'],dca['Treat none'],label='Treat none',linestyle=':')
plt.xlabel('Decision threshold'); plt.ylabel('Net benefit')
plt.title('Exploratory decision-curve analysis'); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'10_decision_curve.png',dpi=180,bbox_inches='tight')
plt.show()
cal_table.to_csv(OUTPUT_ROOT/'calibration_table.csv',index=False)
dca.to_csv(OUTPUT_ROOT/'decision_curve.csv',index=False)

## 12. Calibration-aware selective prediction

Several uncertainty scores compete on calibration patients. The uncertainty method and cutoff are frozen before the locked test is evaluated. This avoids selecting the abstention behavior on the same mistakes used to report its performance.

In [ ]:
def normalized_entropy(p):
    p=np.clip(np.asarray(p),1e-7,1-1e-7)
    return -(p*np.log(p)+(1-p)*np.log(1-p))/np.log(2)


def robust_01(x,ref=None):
    x=np.asarray(x); r=x if ref is None else np.asarray(ref)
    lo,hi=np.quantile(r,[0.01,0.99])
    return np.clip((x-lo)/(hi-lo+1e-12),0,1)


def risk_coverage(y,p,u,threshold,grid=np.linspace(0.20,1.0,81)):
    order=np.argsort(u); rows=[]
    for cov in grid:
        k=max(2,int(round(cov*len(y))))
        idx=order[:k]
        pred=p[idx]>=threshold
        rows.append({
            'Coverage':k/len(y),
            'Risk':1-accuracy_score(y[idx],pred),
            'Accuracy':accuracy_score(y[idx],pred),
            'Brier':brier_score_loss(y[idx],p[idx])
        })
    return pd.DataFrame(rows)

iso_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scale',RobustScaler()),
    ('iso',IsolationForest(n_estimators=250,random_state=SEED,n_jobs=-1,contamination='auto')),
])
iso_pipe.fit(X_train)
ood_cal=-iso_pipe.decision_function(X_cal)
ood_test=-iso_pipe.decision_function(X_test)

dis_cal=Pcal.values.std(axis=1); dis_test=Ptest.values.std(axis=1)
ent_cal=normalized_entropy(p_cal_final); ent_test=normalized_entropy(p_test_final)

uncertainty_cal={
    'Probability margin':1-np.abs(2*p_cal_final-1),
    'Entropy':ent_cal,
    'Model disagreement':dis_cal,
    'Entropy + disagreement':0.75*ent_cal+0.25*robust_01(dis_cal),
    'Entropy + disagreement + OOD':0.65*ent_cal+0.20*robust_01(dis_cal)+0.15*robust_01(ood_cal),
}
uncertainty_test={
    'Probability margin':1-np.abs(2*p_test_final-1),
    'Entropy':ent_test,
    'Model disagreement':dis_test,
    'Entropy + disagreement':0.75*ent_test+0.25*robust_01(dis_test,dis_cal),
    'Entropy + disagreement + OOD':0.65*ent_test+0.20*robust_01(dis_test,dis_cal)+0.15*robust_01(ood_test,ood_cal),
}

unc_rows=[]; cal_curves={}
for name,u in uncertainty_cal.items():
    curve=risk_coverage(y_cal,p_cal_final,u,DECISION_THRESHOLD)
    aurc=np.trapz(curve['Risk'],curve['Coverage'])/(curve['Coverage'].max()-curve['Coverage'].min())
    mistake=(p_cal_final>=DECISION_THRESHOLD).astype(int)!=y_cal
    detection=roc_auc_score(mistake.astype(int),u) if mistake.any() and (~mistake).any() else np.nan
    unc_rows.append({'Uncertainty':name,'Calibration AURC':aurc,'Mistake-detection AUC':detection})
    cal_curves[name]=curve
unc_compare=pd.DataFrame(unc_rows).sort_values(['Calibration AURC','Mistake-detection AUC'],ascending=[True,False])
display(unc_compare)
CHOSEN_UNCERTAINTY=str(unc_compare.iloc[0]['Uncertainty'])

u_cal=uncertainty_cal[CHOSEN_UNCERTAINTY]
u_test=uncertainty_test[CHOSEN_UNCERTAINTY]
uncertainty_cutoff=float(np.quantile(u_cal,TARGET_SELECTIVE_COVERAGE))
accepted=u_test<=uncertainty_cutoff

selective=pd.DataFrame([
    {'Evaluation':'All locked-test patients','Coverage':1.0,**metric_row(y_test,p_test_final,DECISION_THRESHOLD)},
    {'Evaluation':'Reported patients','Coverage':accepted.mean(),**metric_row(y_test[accepted],p_test_final[accepted],DECISION_THRESHOLD)},
])
display(selective)

plt.figure(figsize=(9,5.5))
for name,curve in cal_curves.items():
    plt.plot(curve['Coverage'],curve['Accuracy'],label=name)
plt.xlabel('Coverage'); plt.ylabel('Accuracy on reported patients')
plt.title('Calibration-set uncertainty-method comparison')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'11_uncertainty_method_comparison.png',dpi=180,bbox_inches='tight')
plt.show()

test_curve=risk_coverage(y_test,p_test_final,u_test,DECISION_THRESHOLD)
plt.figure(figsize=(7,5))
plt.plot(test_curve['Coverage'],test_curve['Accuracy'])
plt.axvline(accepted.mean(),linestyle='--',label=f'Frozen operating point ({accepted.mean():.1%})')
plt.xlabel('Coverage'); plt.ylabel('Accuracy on reported patients')
plt.title(f'Locked-test risk–coverage curve\n{CHOSEN_UNCERTAINTY}')
plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'12_locked_test_risk_coverage.png',dpi=180,bbox_inches='tight')
plt.show()

unc_compare.to_csv(OUTPUT_ROOT/'uncertainty_selection.csv',index=False)
selective.to_csv(OUTPUT_ROOT/'selective_prediction_locked_test.csv',index=False)
test_curve.to_csv(OUTPUT_ROOT/'risk_coverage_locked_test.csv',index=False)

## 13. Frozen-model reliability stress tests

All perturbations are applied after the model, parameters, blend, calibration, threshold, and abstention choices are frozen.

In [ ]:
def predict_selected(frame):
    matrix=np.column_stack([fitted[n].predict_proba(frame[FEATURES])[:,1] for n in selected_names])
    raw=matrix@final_weights
    return apply_calibration(calibration_spec,raw)


def evaluate_scenario(name,frame):
    p=predict_selected(frame)
    return {'Scenario':name,**metric_row(y_test,p,DECISION_THRESHOLD)}

rng=np.random.default_rng(SEED)
stress=[evaluate_scenario('Clean locked test',test_df.copy())]

missing=test_df.copy()
missing.loc[:,FEATURES]=missing[FEATURES].mask(rng.random((len(missing),len(FEATURES)))<0.10)
stress.append(evaluate_scenario('10% missing-at-random',missing))

bp=test_df.copy()
bp['ap_hi']=bp['ap_hi']+rng.normal(0,10,len(bp))
bp['ap_lo']=bp['ap_lo']+rng.normal(0,6,len(bp))
bp['pulse_pressure']=bp['ap_hi']-bp['ap_lo']
bp['map']=(bp['ap_hi']+2*bp['ap_lo'])/3
bp['bp_ratio']=bp['ap_hi']/bp['ap_lo']
bp['hypertension_flag']=((bp['ap_hi']>=140)|(bp['ap_lo']>=90)).astype(int)
bp['age_sbp_interaction']=bp['age_years']*bp['ap_hi']/1000.0
bp['combined_risk_count']=(bp['chol_high']+bp['gluc_high']+bp['hypertension_flag']+bp['smoke']+(1-bp['active']))
stress.append(evaluate_scenario('Blood-pressure measurement noise',bp))

age=test_df.copy()
age['age_years']=(age['age_years']+10).clip(18,100)
age['age_sq']=(age['age_years']/10.0)**2
age['age_sbp_interaction']=age['age_years']*age['ap_hi']/1000.0
age['age_bmi_interaction']=age['age_years']*age['bmi']/1000.0
stress.append(evaluate_scenario('Population age +10 years',age))

lifestyle=test_df.copy()
lifestyle[['smoke','alco','active']]=np.nan
lifestyle[['lifestyle_risk_count','combined_risk_count']]=np.nan
stress.append(evaluate_scenario('Lifestyle fields unavailable',lifestyle))

stress=pd.DataFrame(stress)
display(stress)

fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,metric in zip(axes,['ROC-AUC','Brier','ECE']):
    s=stress[['Scenario',metric]].sort_values(metric)
    ax.barh(s['Scenario'],s[metric]); ax.set_title(metric)
plt.suptitle('Frozen-model reliability stress tests')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'13_reliability_stress_tests.png',dpi=180,bbox_inches='tight')
plt.show()
stress.to_csv(OUTPUT_ROOT/'stress_test_results.csv',index=False)

## 14. Patient subgroup operating characteristics

Subgroup results are descriptive. They do not establish fairness or deployment suitability. Confidence intervals are patient bootstraps within each subgroup.

In [ ]:
audit=test_df[[GROUP,'gender','age_years','cholesterol','ap_hi',TARGET]].copy()
audit['p']=p_test_final
audit['pred']=(p_test_final>=DECISION_THRESHOLD).astype(int)
audit['Recorded sex']=np.where(audit['gender']==2,'Male','Female')
audit['Age group']=pd.cut(audit['age_years'],[-np.inf,49,59,69,np.inf],labels=['<50','50–59','60–69','70+'])
audit['BP group']=pd.cut(audit['ap_hi'],[-np.inf,119,139,159,np.inf],labels=['<120','120–139','140–159','160+'])


def subgroup_table(col):
    rows=[]
    for group,d in audit.groupby(col,observed=True):
        if len(d)<50 or d[TARGET].nunique()<2:
            continue
        auc=roc_auc_score(d[TARGET],d['p'])
        lo,hi=bootstrap_metric_ci(d[TARGET].to_numpy(),d['p'].to_numpy(),roc_auc_score,n_boot=min(350,BOOTSTRAPS),seed=SEED+len(rows)+40)
        rows.append({
            'Dimension':col,'Group':str(group),'N':len(d),'Prevalence':d[TARGET].mean(),
            'ROC-AUC':auc,'AUC CI low':lo,'AUC CI high':hi,
            'Brier':brier_score_loss(d[TARGET],d['p']),'ECE':ece_score(d[TARGET],d['p']),
            'Sensitivity':recall_score(d[TARGET],d['pred'],zero_division=0),
            'Specificity':recall_score(1-d[TARGET],1-d['pred'],zero_division=0),
        })
    return pd.DataFrame(rows)

subgroups=pd.concat([subgroup_table(c) for c in ['Recorded sex','Age group','BP group']],ignore_index=True)
display(subgroups)

plot=subgroups.copy(); plot['Label']=plot['Dimension']+' — '+plot['Group']
plot=plot.sort_values('ROC-AUC')
plt.figure(figsize=(9,7))
xerr=np.vstack([plot['ROC-AUC']-plot['AUC CI low'],plot['AUC CI high']-plot['ROC-AUC']])
plt.errorbar(plot['ROC-AUC'],plot['Label'],xerr=xerr,fmt='o',capsize=3)
plt.axvline(final_metrics['ROC-AUC'],linestyle='--',label='Overall locked-test AUC')
plt.xlabel('ROC-AUC (95% patient bootstrap CI)')
plt.title('Patient subgroup operating characteristics')
plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'14_subgroup_forest.png',dpi=180,bbox_inches='tight')
plt.show()
subgroups.to_csv(OUTPUT_ROOT/'subgroup_audit.csv',index=False)

## 15. Interpretation: permutation importance and optional SHAP

Importance describes model behavior, not causal medical effects.

In [ ]:
lead_model=weight_table.iloc[0]['Model']
lead_est=fitted[lead_model]
sample_idx=np.random.default_rng(SEED).choice(len(test_df),size=min(3000,len(test_df)),replace=False)
perm=permutation_importance(
    lead_est,X_test.iloc[sample_idx],y_test[sample_idx],scoring='roc_auc',
    n_repeats=5,random_state=SEED,n_jobs=-1
)
importance=pd.DataFrame({'Feature':FEATURES,'Importance':perm.importances_mean,'Std':perm.importances_std}).sort_values('Importance',ascending=False)
display(importance.head(18))

p=importance.head(18).sort_values('Importance')
plt.figure(figsize=(8,7))
plt.barh(p['Feature'],p['Importance'],xerr=p['Std'])
plt.xlabel('Mean decrease in ROC-AUC')
plt.title(f'Permutation importance — {lead_model}')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'15_permutation_importance.png',dpi=180,bbox_inches='tight')
plt.show()
importance.to_csv(OUTPUT_ROOT/'feature_importance.csv',index=False)

try:
    if not ENABLE_SHAP:
        raise RuntimeError('SHAP disabled by configuration')
    import shap
    pipe=lead_est
    estimator=pipe.named_steps['model'] if hasattr(pipe,'named_steps') else pipe
    transformed=pipe.named_steps['imputer'].transform(X_test.iloc[sample_idx[:1000]]) if hasattr(pipe,'named_steps') and 'imputer' in pipe.named_steps else X_test.iloc[sample_idx[:1000]].values
    explainer=shap.Explainer(estimator,transformed[:200])
    sv=explainer(transformed[:1000])
    shap.plots.beeswarm(sv,max_display=15,show=False)
    plt.title(f'SHAP behavior summary — {lead_model}')
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT/'16_shap_summary.png',dpi=180,bbox_inches='tight')
    plt.show()
except Exception as e:
    print('SHAP plot skipped safely:',type(e).__name__,str(e)[:200])

## 16. Source-only model selection and second-dataset transportability

This is the central external experiment.

Portable model families use only concepts that can be aligned without the second dataset’s labels. Their parameters and ranking are selected solely from source development patients. After those choices are frozen, every portable family is evaluated on both the source locked test and the second dataset.

The externally best-looking family is **not** retroactively selected. This prevents post-hoc external-test optimization and makes model-ranking instability visible.

In [ ]:
source_portable=pd.DataFrame({
    'age':clean['age_years'].astype(float),
    'male':(clean['gender']==2).astype(int),
    'systolic_bp':clean['ap_hi'].astype(float),
    'high_cholesterol':(clean['cholesterol']>1).astype(int),
    'high_glucose':(clean['gluc']>1).astype(int),
    'target':clean[TARGET].astype(int),
    'id':clean[GROUP],
})

sex_col=next((c for c in heart.columns if c.lower() in {'sex_m','sex'}),None)
fast_col=next((c for c in heart.columns if c.lower() in {'fastingbs','fasting_bs'}),None)
if sex_col is None:
    raise KeyError('Could not identify sex field in second table.')

external_portable=pd.DataFrame({
    'age':heart['Age'].astype(float),
    'male':heart[sex_col].astype(int),
    'systolic_bp':heart['RestingBP'].replace(0,np.nan).astype(float),
    'high_cholesterol':(heart['Cholesterol'].replace(0,np.nan)>=200).astype(float),
    'high_glucose':heart[fast_col].astype(float) if fast_col else 0.0,
    'target':heart['HeartDisease'].astype(int),
})
PORTABLE=['age','male','systolic_bp','high_cholesterol','high_glucose']

source_train=source_portable[source_portable.id.isin(set(train_df[GROUP]))].reset_index(drop=True)
source_cal=source_portable[source_portable.id.isin(set(cal_df[GROUP]))].reset_index(drop=True)
source_test=source_portable[source_portable.id.isin(set(test_df[GROUP]))].reset_index(drop=True)

portable_base={
    'Portable logistic':scaled(LogisticRegression(max_iter=2000,C=0.5,random_state=SEED)),
    'Portable LDA':scaled(LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto')),
    'Portable HistGB':unscaled(HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_leaf_nodes=16,min_samples_leaf=30,l2_regularization=2,random_state=SEED)),
}
portable_space={
    'Portable logistic':{'model__C':np.logspace(-3,2,20).tolist(),'model__class_weight':[None,'balanced']},
    'Portable LDA':{'model__shrinkage':['auto']+np.linspace(0,1,15).tolist()},
    'Portable HistGB':{
        'model__max_iter':[200,350,500], 'model__learning_rate':[0.02,0.04,0.07],
        'model__max_leaf_nodes':[7,15,31], 'model__min_samples_leaf':[15,30,60],
        'model__l2_regularization':[0.5,2,5]
    },
}
if OPTIONAL.get('XGBoost'):
    portable_base['Portable XGBoost']=unscaled(XGBClassifier(
        n_estimators=450,max_depth=3,learning_rate=0.04,min_child_weight=5,
        subsample=0.85,colsample_bytree=0.9,reg_lambda=3,objective='binary:logistic',
        eval_metric='logloss',tree_method='hist',n_jobs=-1,random_state=SEED
    ))
    portable_space['Portable XGBoost']={
        'model__n_estimators':[300,450,650], 'model__max_depth':[2,3,4,5],
        'model__learning_rate':[0.02,0.04,0.07], 'model__min_child_weight':[1,3,5,8],
        'model__subsample':[0.75,0.9,1.0], 'model__colsample_bytree':[0.75,0.9,1.0],
        'model__reg_lambda':[1,3,7]
    }
if OPTIONAL.get('LightGBM'):
    portable_base['Portable LightGBM']=unscaled(LGBMClassifier(
        n_estimators=450,learning_rate=0.04,num_leaves=15,min_child_samples=35,
        n_jobs=-1,random_state=SEED,verbosity=-1
    ))
    portable_space['Portable LightGBM']={
        'model__n_estimators':[300,450,650], 'model__learning_rate':[0.02,0.04,0.07],
        'model__num_leaves':[7,15,24,31], 'model__max_depth':[-1,3,5,7],
        'model__min_child_samples':[15,35,70], 'model__reg_lambda':[0,2,5]
    }
if OPTIONAL.get('CatBoost'):
    portable_base['Portable CatBoost']=CatBoostCompat(iterations=450,depth=5,learning_rate=0.04,l2_leaf_reg=5,random_seed=SEED)
    portable_space['Portable CatBoost']={
        'iterations':[300,450,650], 'depth':[4,5,6], 'learning_rate':[0.02,0.04,0.07],
        'l2_leaf_reg':[2,5,10], 'random_strength':[0,1,2], 'bagging_temperature':[0,1,2]
    }

Xp=source_train[PORTABLE]
yp=source_train.target.to_numpy()
gp=source_train.id.to_numpy()
port_splits=list(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED+61).split(Xp,yp,gp))
portable_trials=[]
portable_best={}
portable_oof={}

for j,(name,base_est) in enumerate(portable_base.items(),1):
    budget=6 if RUN_MODE=='competition' else 2
    candidates=[{}]+list(ParameterSampler(portable_space[name],n_iter=budget,random_state=SEED+60+j))
    family=[]
    for trial,params in enumerate(candidates):
        try:
            est=clone(base_est).set_params(**params)
            oof,_=grouped_oof_for_estimator(est,Xp,yp,gp,port_splits)
            m=metric_row(yp,oof); obj=hpo_objective(m)
            row={'Model':name,'Trial':trial,'Status':'ok','Objective':obj,'Parameters':json.dumps(jsonable_params(params),sort_keys=True),**m}
        except Exception as e:
            row={'Model':name,'Trial':trial,'Status':'failed','Objective':np.nan,'Parameters':json.dumps(jsonable_params(params),sort_keys=True),'Error':f'{type(e).__name__}: {str(e)[:200]}'}
        portable_trials.append(row); family.append(row)
    ok=pd.DataFrame(family); ok=ok[ok.Status=='ok'].sort_values(['Objective','ROC-AUC'],ascending=[False,False])
    win=ok.iloc[0]
    portable_best[name]=json.loads(win['Parameters'])
    best_est=clone(base_est).set_params(**portable_best[name])
    portable_oof[name],_=grouped_oof_for_estimator(best_est,Xp,yp,gp,port_splits)

portable_selection=[]
external_sweep=[]
source_fit=pd.concat([source_train,source_cal],ignore_index=True)
for name,base_est in portable_base.items():
    oof=portable_oof[name]
    selection_row={'Model':name,**metric_row(yp,oof)}
    selection_row['Objective']=hpo_objective(selection_row)
    portable_selection.append(selection_row)

    final_est=clone(base_est).set_params(**portable_best[name])
    final_est.fit(source_fit[PORTABLE],source_fit.target)
    source_p_each=final_est.predict_proba(source_test[PORTABLE])[:,1]
    ext_p_each=final_est.predict_proba(external_portable[PORTABLE])[:,1]
    external_sweep.append({
        'Model':name,
        'Source OOF ROC-AUC':roc_auc_score(yp,oof),
        'Source locked ROC-AUC':roc_auc_score(source_test.target,source_p_each),
        'External ROC-AUC':roc_auc_score(external_portable.target,ext_p_each),
        'External PR-AUC':average_precision_score(external_portable.target,ext_p_each),
        'External Brier':brier_score_loss(external_portable.target,ext_p_each),
        'External ECE':ece_score(external_portable.target,ext_p_each),
    })

portable_selection=pd.DataFrame(portable_selection).sort_values(['Objective','ROC-AUC'],ascending=[False,False])
external_sweep=pd.DataFrame(external_sweep)
display(portable_selection)
display(external_sweep.sort_values('Source OOF ROC-AUC',ascending=False))

portable_name=str(portable_selection.iloc[0]['Model'])
portable_final=clone(portable_base[portable_name]).set_params(**portable_best[portable_name])
portable_final.fit(source_fit[PORTABLE],source_fit.target)
source_p=portable_final.predict_proba(source_test[PORTABLE])[:,1]
external_p=portable_final.predict_proba(external_portable[PORTABLE])[:,1]

source_auc_ci=bootstrap_metric_ci(source_test.target.to_numpy(),source_p,roc_auc_score,n_boot=max(BOOTSTRAPS,700),seed=SEED+71)
ext_auc_ci=bootstrap_metric_ci(external_portable.target.to_numpy(),external_p,roc_auc_score,n_boot=max(BOOTSTRAPS,900),seed=SEED+72)
transport=pd.DataFrame([
    {'Dataset':'Source locked test','N':len(source_test),'Prevalence':source_test.target.mean(),**metric_row(source_test.target,source_p),'AUC CI low':source_auc_ci[0],'AUC CI high':source_auc_ci[1]},
    {'Dataset':'Second provided dataset','N':len(external_portable),'Prevalence':external_portable.target.mean(),**metric_row(external_portable.target,external_p),'AUC CI low':ext_auc_ci[0],'AUC CI high':ext_auc_ci[1]},
])
display(transport)

fig,axes=plt.subplots(1,2,figsize=(13,5))
xx=np.arange(len(transport)); yy=transport['ROC-AUC'].to_numpy()
err=np.vstack([yy-transport['AUC CI low'],transport['AUC CI high']-yy])
axes[0].errorbar(xx,yy,yerr=err,fmt='o',capsize=5)
axes[0].set_xticks(xx,transport['Dataset']); axes[0].set_ylim(0.45,0.90)
axes[0].set_ylabel('ROC-AUC'); axes[0].set_title(f'Frozen source-selected model: {portable_name}')
axes[1].scatter(external_sweep['Source OOF ROC-AUC'],external_sweep['External ROC-AUC'])
for _,r in external_sweep.iterrows():
    axes[1].annotate(r['Model'].replace('Portable ',''),(r['Source OOF ROC-AUC'],r['External ROC-AUC']),fontsize=8)
axes[1].axhline(0.5,linestyle='--',linewidth=1)
axes[1].set_xlabel('Source development OOF ROC-AUC'); axes[1].set_ylabel('External ROC-AUC')
axes[1].set_title('Model ranking stability across datasets')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT/'17_transportability_and_model_instability.png',dpi=180,bbox_inches='tight')
plt.show()

source_domain=source_test[PORTABLE].copy(); source_domain['domain']=0
external_domain=external_portable[PORTABLE].copy(); external_domain['domain']=1
domain=pd.concat([source_domain,external_domain],ignore_index=True)
idx_tr,idx_va=train_test_split(np.arange(len(domain)),test_size=0.30,stratify=domain.domain,random_state=SEED)
domain_model=unscaled(HistGradientBoostingClassifier(max_iter=180,max_leaf_nodes=16,learning_rate=0.05,random_state=SEED))
domain_model.fit(domain.iloc[idx_tr][PORTABLE],domain.iloc[idx_tr].domain)
domain_p=domain_model.predict_proba(domain.iloc[idx_va][PORTABLE])[:,1]
domain_auc=roc_auc_score(domain.iloc[idx_va].domain,domain_p)
print(f'Domain-classifier AUC: {domain_auc:.4f} (0.50=no detectable shift; 1.00=fully separable)')

shift_rows=[]
for f in PORTABLE:
    s=source_test[f].dropna(); e=external_portable[f].dropna()
    shift_rows.append({'Feature':f,'Source mean':s.mean(),'External mean':e.mean(),'Standardized mean difference':(e.mean()-s.mean())/np.sqrt((s.var()+e.var())/2+1e-12)})
shift=pd.DataFrame(shift_rows)
display(shift)

pd.DataFrame(portable_trials).to_csv(OUTPUT_ROOT/'portable_hpo_trials.csv',index=False)
portable_selection.to_csv(OUTPUT_ROOT/'portable_model_selection.csv',index=False)
external_sweep.to_csv(OUTPUT_ROOT/'portable_external_model_sweep.csv',index=False)
transport.to_csv(OUTPUT_ROOT/'transportability_results.csv',index=False)
shift.to_csv(OUTPUT_ROOT/'domain_shift_features.csv',index=False)
with open(OUTPUT_ROOT/'portable_best_parameters.json','w',encoding='utf-8') as f:
    json.dump(portable_best,f,indent=2)

## 17. ECG data-readiness and linkage audit

The ECG matrix is intentionally not forced into the supervised model unless it provides a defensible patient identifier or target linkage. Omitting an unlinked modality is scientifically stronger than manufacturing an invalid multimodal result.

In [ ]:
ecg_summary=[]
if ecg_path is not None:
    # The ECG file is extremely wide, so detect its delimiter once and read only
    # the header plus three rows. It is not used in the supervised tabular model.
    ecg_spec = detect_csv_spec(ecg_path)
    header = smart_read_csv(ecg_path, nrows=0).columns.tolist()
    preview = smart_read_csv(ecg_path, nrows=3)
    candidate_link_cols=[c for c in header if c.lower() in {'id','patient_id','subject_id','cardio','target','label','heartdisease'}]
    numeric_cols=[c for c in preview.columns if c!='Unnamed: 0' and pd.api.types.is_numeric_dtype(preview[c])]
    print('ECG separator:', repr(ecg_spec['sep']), 'encoding:', ecg_spec['encoding'])
    print('ECG columns:',len(header))
    print('Potential linkage/label columns:',candidate_link_cols)
    print('Conclusion:', 'linkage found' if candidate_link_cols else 'no defensible patient/target linkage found')
    ecg_summary=[
        {'Property':'Columns','Value':len(header)},
        {'Property':'Preview rows read','Value':len(preview)},
        {'Property':'Separator','Value':repr(ecg_spec['sep'])},
        {'Property':'Encoding','Value':ecg_spec['encoding']},
        {'Property':'Potential linkage/label columns','Value':', '.join(candidate_link_cols) if candidate_link_cols else 'None'},
        {'Property':'Used in supervised final model','Value':'No'},
    ]
    if numeric_cols:
        trace=pd.to_numeric(preview.iloc[0][numeric_cols[:2500]],errors='coerce').to_numpy(float)
        plt.figure(figsize=(12,3.5)); plt.plot(trace)
        plt.xlabel('Sample index'); plt.ylabel('Recorded value')
        plt.title('Representative ECG numeric trace (data-readiness audit only)')
        plt.tight_layout(); plt.savefig(OUTPUT_ROOT/'15_ecg_trace.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    ecg_summary=[{'Property':'ECG file','Value':'Not found'}]
pd.DataFrame(ecg_summary).to_csv(OUTPUT_ROOT/'ecg_readiness_audit.csv',index=False)
display(pd.DataFrame(ecg_summary))


## 18. Save the frozen evidence bundle and write-up metrics

In [ ]:
import joblib

bundle={
    'features':FEATURES,
    'selected_model_names':selected_names,
    'models':fitted,
    'ensemble_weights':final_weights,
    'calibration_spec':calibration_spec,
    'decision_threshold':DECISION_THRESHOLD,
    'uncertainty_method':CHOSEN_UNCERTAINTY,
    'uncertainty_cutoff':uncertainty_cutoff,
    'target_selective_coverage':TARGET_SELECTIVE_COVERAGE,
    'hpo_best_parameters':best_params,
    'quality_ranges':{
        'age_years':[18,100],'height':[130,210],'weight':[35,200],
        'ap_hi':[70,250],'ap_lo':[40,150],'bmi':[12,70]
    },
    'seed':SEED,
}

# This dump contains no lambda/local calibration function and is therefore reproducible.
joblib.dump(bundle,OUTPUT_ROOT/'pulseproof_patientlevel_hpo_bundle.joblib',compress=3)

summary={
    'rows_total':int(len(data)),
    'rows_after_quality_gate':int(len(clean)),
    'unique_patients':int(clean[GROUP].nunique()),
    'max_rows_per_patient':int(clean.groupby(GROUP).size().max()),
    'patient_overlap_across_locked_splits':int(overlap['Patient overlap'].sum()),
    'development_patients':int(train_df[GROUP].nunique()),
    'calibration_patients':int(cal_df[GROUP].nunique()),
    'locked_test_patients':int(test_df[GROUP].nunique()),
    'hpo_models':int(hpo_summary.Model.nunique()),
    'hpo_successful_trials':int((hpo_trials.Status=='ok').sum()),
    'best_tuned_oof_model':str(benchmark.iloc[0]['Model']),
    'best_tuned_oof_auc':float(benchmark.iloc[0]['ROC-AUC']),
    'chosen_blend':str(CHOSEN_BLEND),
    'chosen_calibration':str(CHOSEN_CAL),
    'decision_threshold':float(DECISION_THRESHOLD),
    'locked_test_auc':float(final_metrics['ROC-AUC']),
    'locked_test_auc_ci':[float(auc_ci[0]),float(auc_ci[1])],
    'locked_test_pr_auc':float(final_metrics['PR-AUC']),
    'locked_test_brier':float(final_metrics['Brier']),
    'locked_test_ece':float(final_metrics['ECE']),
    'locked_test_balanced_accuracy':float(final_metrics['Balanced accuracy']),
    'uncertainty_method':str(CHOSEN_UNCERTAINTY),
    'selective_coverage':float(accepted.mean()),
    'selective_accuracy':float(accuracy_score(y_test[accepted],p_test_final[accepted]>=DECISION_THRESHOLD)),
    'source_portable_model':str(portable_name),
    'source_portable_auc':float(transport.loc[transport.Dataset=='Source locked test','ROC-AUC'].iloc[0]),
    'external_auc':float(transport.loc[transport.Dataset=='Second provided dataset','ROC-AUC'].iloc[0]),
    'external_auc_ci':[float(ext_auc_ci[0]),float(ext_auc_ci[1])],
    'domain_classifier_auc':float(domain_auc),
}
with open(OUTPUT_ROOT/'final_metrics.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

writeup=f"""## Verified patient-level HPO notebook results

- Patients analyzed after quality gate: **{summary['unique_patients']:,}**
- Maximum rows per patient: **{summary['max_rows_per_patient']}**
- Patient overlap across development/calibration/test: **{summary['patient_overlap_across_locked_splits']}**
- Protocol: **70% development / 10% calibration / 20% locked test**
- Model families searched: **{summary['hpo_models']}**
- Successful HPO configurations: **{summary['hpo_successful_trials']}**
- Best tuned five-fold patient-grouped OOF model: **{summary['best_tuned_oof_model']}** (AUC **{summary['best_tuned_oof_auc']:.4f}**)
- Frozen ensemble: **{summary['chosen_blend']}**, calibration: **{summary['chosen_calibration']}**
- Calibration-selected decision threshold: **{summary['decision_threshold']:.3f}**
- Locked-test ROC-AUC: **{summary['locked_test_auc']:.4f}** (95% CI **{summary['locked_test_auc_ci'][0]:.4f}–{summary['locked_test_auc_ci'][1]:.4f}**)
- Locked-test PR-AUC: **{summary['locked_test_pr_auc']:.4f}**
- Locked-test Brier / ECE: **{summary['locked_test_brier']:.4f} / {summary['locked_test_ece']:.4f}**
- Frozen selective coverage: **{summary['selective_coverage']:.1%}**
- Accuracy on reported patients: **{summary['selective_accuracy']:.1%}**
- Source-selected portable model: **{summary['source_portable_model']}**
- Source locked portable AUC: **{summary['source_portable_auc']:.4f}**
- Second-dataset AUC: **{summary['external_auc']:.4f}** (95% CI **{summary['external_auc_ci'][0]:.4f}–{summary['external_auc_ci'][1]:.4f}**)
- Domain-classifier AUC: **{summary['domain_classifier_auc']:.4f}**

The second-dataset experiment is a source-only transportability stress test, not external clinical validation.
"""
(OUTPUT_ROOT/'WRITEUP_METRICS.md').write_text(writeup,encoding='utf-8')
display(Markdown(writeup))
print('All outputs saved under:',OUTPUT_ROOT)

## 19. Artifact inventory

In [ ]:
artifacts=[]
for p in sorted(OUTPUT_ROOT.iterdir()):
    if p.is_file():
        artifacts.append({'File':p.name,'Size KB':round(p.stat().st_size/1024,1)})
artifacts=pd.DataFrame(artifacts)
display(artifacts)
artifacts.to_csv(OUTPUT_ROOT/'artifact_inventory.csv',index=False)

## Conclusions and research contribution

PulseProof is not presented as a deployment-ready diagnostic model. Its contribution is an auditable evaluation architecture:

- patient identity is verified before splitting,
- all model and parameter choices are confined to development patients,
- tuned models are compared under identical patient-grouped folds,
- OOF predictions—not locked-test labels—determine ensemble weights,
- calibration patients determine probability calibration, operating threshold, and abstention behavior,
- the internal test is opened once,
- the second dataset is never used for source-model selection,
- external ranking instability and dataset shift are reported rather than concealed,
- ECG is excluded from supervised modeling when no defensible linkage is available.

### Methodological references

1. Collins GS et al. **TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods.** BMJ. 2024;385:e078378.
2. Vickers AJ, Elkin EB. **Decision curve analysis: a novel method for evaluating prediction models.** Medical Decision Making. 2006;26(6):565–574.
3. Scikit-learn documentation: `StratifiedGroupKFold`, `ParameterSampler`, calibration, and probability metrics.
4. XGBoost, LightGBM, and CatBoost official parameter-tuning documentation.

> The strongest scientific message may not be the last decimal place of internal AUC. It may be the demonstrated gap between internal validation and cross-dataset transportability—and the disciplined protocol that makes that gap believable.

**Loader revision:** V7 uses delimiter- and encoding-aware parsing for all source CSV files.